# Foundations of Probability and Information Theory

> From Cross-Entropy during training to sampling during generation, a language model always manipulates probability distributions. For a four-token vocabulary it might output `[0.3, 0.2, 0.4, 0.1]`: every value is between 0 and 1 and the values sum to 1.
>
> **Softmax** converts logits to probabilities, and sampling chooses a Token according to them. **Cross-Entropy** measures the gap between the model distribution and the correct answer. **Entropy** measures uncertainty, **KL Divergence** compares distributions, and **Perplexity** converts average Cross-Entropy into a more intuitive scale.

Raw logits such as `[2.0, 1.0, 0.1, -1.0]` may be positive or negative and need not sum to 1. This appendix follows the path raw scores → probability distribution → information measures, building the mathematics reused by training, decoding, MoE routing, and alignment.


## 1. Model Output Probabilities

For a vocabulary of size $V$, the model produces a vector $p=(p_1,\ldots,p_V)$ at each position, where every probability is nonnegative and their sum is 1. This is a **categorical distribution**: a probability assigned to each member of a finite set.

For example, logits $z=[2.0,1.0,0.1,-1.0]$ uniquely determine one categorical distribution. Expert routing in MoE has the same mathematical form: next-token prediction assigns probabilities to tokens, while routing assigns probabilities to experts.


In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Demonstrate a categorical distribution with tiny numbers
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
print("logits:           ", logits.tolist())
print("Exponentiated exp(logits):", [round(x, 4) for x in torch.exp(logits).tolist()])
print("Normalized probabilities: ", [round(x, 4) for x in F.softmax(logits, dim=-1).tolist()])
print("Probability sum:           ", round(F.softmax(logits, dim=-1).sum().item(), 6))
print()
print("Key observation: logits can be any real values; softmax converts them into nonnegative probabilities that sum to one.")
print("This is exactly the categorical parameterization: V classes, each with one probability.")


### 1.1 From Logits to Probabilities

Logits are unrestricted real-valued scores. Softmax converts them to probabilities:

$$p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}.$$

Exponentials are positive, and division by their sum normalizes the result. Softmax also preserves order: $z_i>z_j$ implies $p_i>p_j$, so greedy decoding can select the largest logit. The differentiable connection between likelihood and Cross-Entropy is why this transformation is central to language-model training.


### 1.2 Numerical Stability of Softmax

The direct formula overflows for a large logit such as 1000. Softmax is unchanged when the same constant is subtracted from every logit:

$$\text{softmax}(z)_i=\frac{e^{z_i-c}}{\sum_j e^{z_j-c}}.$$

Choose $c=\max_j z_j$. The largest exponential becomes $e^0=1$ and all others are at most 1. This yields

$$\text{logsumexp}(z)=c+\log\sum_j e^{z_j-c},\qquad
\log p_i=z_i-\text{logsumexp}(z).$$

This is the starting point of online softmax in FlashAttention. PyTorch uses the same stable calculation in `F.log_softmax` and `F.cross_entropy`.


In [ ]:
# Compare naive softmax with numerically stable softmax
big_logits = torch.tensor([1000.0, 1001.0, 1002.0])

naive_exp = torch.exp(big_logits)
print("Interpretation:")
print("  exp([1000, 1001, 1002]) =", naive_exp.tolist())
print("  -> Every value overflows to inf, so normalization is impossible")
print()

# Subtract the maximum first
stable_shifted = big_logits - big_logits.max()
stable_exp = torch.exp(stable_shifted)
stable_probs = stable_exp / stable_exp.sum()
print("Numerically stable version, subtracting max=1002:")
print("  shifted logits =", stable_shifted.tolist())
print("  exp(shifted)   =", [round(x, 6) for x in stable_exp.tolist()])
print("  softmax        =", [round(x, 4) for x in stable_probs.tolist()])
print()

# PyTorch already does this internally
torch_logsoftmax = F.log_softmax(big_logits, dim=-1)
print("PyTorch log_softmax:", [round(x, 4) for x in torch_logsoftmax.tolist()])
print("Key observation: subtracting the maximum leaves softmax unchanged while preventing exp overflow.")


## 2. Sampling from a Distribution

During inference, the model samples a token from a categorical distribution. Temperature, top-k, and top-p first reshape that distribution and then sample from it. The next cells apply all three methods to the same logits.


### 2.1 Temperature and Randomness

Temperature sampling uses $p^{(T)}=\text{softmax}(z/T)$. Temperature divides logits, not probabilities.

- As $T\to0^+$, the largest scaled logit dominates and the distribution approaches one-hot, like greedy decoding.
- As $T\to\infty$, every scaled logit approaches zero and the distribution approaches uniform.
- At $T=1$, the original softmax distribution is unchanged.

Temperature is therefore a continuous control from highly deterministic to highly random sampling.


In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
print(f"{'student logp':>12} {'teacher logp':>12} {'k1':>10} {'k2':>10} {'k3':>10}  Note")
print("-" * 70)

for T in [0.1, 0.5, 1.0, 2.0, 10.0, 1000.0]:
    probs = F.softmax(logits / T, dim=-1)
    row = f"{T:>5.1f}  " + "  ".join(f"{p:>7.4f}" for p in probs.tolist())
    if T <= 0.1:
        note = "near argmax (greedy)"
    elif T == 1.0:
        note = "Both pass through"
    elif T >= 1000.0:
        note = "near uniform"
    else:
        note = ""
    print(row + "  " + note)

print()
print("Key observation: temperature is continuous. As T approaches zero the distribution becomes one-hot; as T grows it approaches uniform.")


### 2.2 Top-k Sampling

Top-k keeps the $k$ highest-probability tokens, removes all others, and renormalizes. If $S_k$ is the retained set,

$$p_i^{(\text{top-}k)}=\begin{cases}
\frac{p_i}{\sum_{j\in S_k}p_j}&i\in S_k\\
0&i\notin S_k.
\end{cases}$$

With $k=1$, this is greedy decoding. Implementations commonly set excluded logits to $-\infty$, making their softmax probabilities zero—the same masking technique used in Attention.


### 2.3 Top-p Sampling

A fixed $k$ ignores whether a distribution is sharp or flat. Top-p instead sorts probabilities from largest to smallest and retains the smallest prefix $S_p$ whose cumulative probability reaches a threshold such as 0.9. It zeroes and renormalizes the rest.

At $p=1$ nothing is removed; a very small $p$ approaches greedy decoding. Hugging Face retains the token that first crosses the threshold, so the retained mass may be slightly above $p$.


In [ ]:
# Use sharp logits to compare outputs under three sampling methods
logits = torch.tensor([3.0, 2.0, 1.0, 0.5, 0.0, -1.0])
base_probs = F.softmax(logits, dim=-1)
print("Original logits:", logits.tolist())
print("Original probabilities:", [round(p, 4) for p in base_probs.tolist()])
print()

# Temperature: T=0.5
probs_t = F.softmax(logits / 0.5, dim=-1)
print("Temperature=0.5, lower and sharper:")
print("  ", [round(p, 4) for p in probs_t.tolist()])
print()

# Top-k=2: keep the two largest
top2_vals, top2_idx = torch.topk(logits, 2)
mask_k = torch.full_like(logits, float('-inf'))
mask_k[top2_idx] = top2_vals
probs_k = F.softmax(mask_k, dim=-1)
print("Top-k=2, truncated to two:")
print("  ", [round(p, 4) if p > 0 else '  0   ' for p in probs_k.tolist()])
print()

# Top-p=0.8: cumulative probability reaches 0.8
sorted_probs, sorted_idx = torch.sort(base_probs, descending=True)
cum = torch.cumsum(sorted_probs, dim=0)
keep_mask = cum <= 0.8
keep_mask[0] = True  # keep at least one
# Also keep the Token that first pushes cumulative mass beyond 0.8
first_exceed = (cum > 0.8).nonzero()
if len(first_exceed) > 0:
    keep_mask[first_exceed[0].item()] = True
kept_idx = sorted_idx[keep_mask]
mask_p = torch.full_like(logits, float('-inf'))
mask_p[kept_idx] = logits[kept_idx]
probs_p = F.softmax(mask_p, dim=-1)
print("Top-p=0.8, truncated by cumulative probability:")
print("  ", [round(p, 4) if p > 0 else '  0   ' for p in probs_p.tolist()])
print()
print("Key observation:")
print("  Temperature changes all relative probabilities without setting any class to zero")
print("  Top-k truncates to a fixed count; Top-p truncates by cumulative probability")
print("  Production commonly combines Top-k and Top-p: cap the count, then adapt to probability mass")


### 2.4 The Gumbel-Max Trick

An equivalent way to sample a categorical distribution is to add independent Gumbel noise $g_i\sim\text{Gumbel}(0,1)$ to every logit and take the argmax. The resulting index follows the original distribution exactly.

This reformulation moves randomness into additive noise. Replacing argmax with a soft approximation then permits gradients to flow through a sampling-like operation, an important reparameterization idea in reinforcement learning. See the [Gumbel-Softmax paper](https://arxiv.org/abs/1611.01144) for the derivation.


## 3. Uncertainty and Differences Between Distributions

We now ask two questions: how uncertain is one distribution, and how different are two distributions? Entropy, Cross-Entropy, and KL Divergence answer them. A fair die and a biased die will make the definitions concrete before we connect them algebraically.


### 3.1 Entropy

Entropy measures the uncertainty within a distribution $p$:

$$H(p)=-\sum_x p(x)\log p(x).$$

The quantity $-\log p(x)$ represents surprise: rare events are more surprising. Entropy is average surprise. A fair six-sided die has maximum entropy, while a die with 99% probability on one face has entropy near zero. For $V$ categories, entropy ranges from 0 for one-hot to $\log V$ for uniform. Low predictive entropy means confidence, not necessarily correctness.


In [ ]:
# Compare entropy using six-sided dice
import math

fair_die = torch.tensor([1/6] * 6)
biased_die = torch.tensor([0.99, 0.002, 0.002, 0.002, 0.002, 0.002])
one_hot_die = torch.tensor([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])

def entropy(p, eps=1e-12):
    """Compute discrete-distribution entropy, adding eps to avoid log(0)."""
    return -(p * torch.log(p + eps)).sum().item()

print(f"Fair-die entropy:    {entropy(fair_die):.4f}  (maximum log(6) = {math.log(6):.4f})")
print(f"Biased-die entropy:  {entropy(biased_die):.4f}  (nearly certain, close to 0)")
print(f"One-hot entropy:     {entropy(one_hot_die):.4f}  (fully certain, equal to 0)")
print()
print("Key observation: the uniform distribution has maximum entropy, while a one-hot distribution has entropy zero.")
print("During LM training, the true answer's one-hot distribution has entropy 0; this fact will recur.")


### 3.2 Cross-Entropy

Cross-Entropy measures the average cost of encoding events from a true distribution $p$ using a predicted distribution $q$:

$$H(p,q)=-\sum_x p(x)\log q(x).$$

The roles differ: $p$ supplies event frequencies and $q$ supplies code lengths. The value is small when $q$ assigns high probability where $p$ does. In language-model training, $p$ is the one-hot target and $q$ is the model prediction, so minimizing Cross-Entropy concentrates probability on the correct token.


### 3.3 KL Divergence

KL Divergence measures how one distribution differs from another:

$$D_{KL}(p\|q)=\sum_x p(x)\log\frac{p(x)}{q(x)}.$$

Remember three properties:

- It is nonnegative and equals zero exactly when $p=q$.
- It is asymmetric, so it is a divergence rather than a distance.
- Direction matters: a position with $p(x)=0$ contributes zero, but $p(x)>0$ and $q(x)=0$ makes the result infinite.

RLHF uses KL to keep a new policy near a reference policy; distillation uses it to make a student match a teacher.


### 3.4 Decomposing Cross-Entropy

Expanding the definition gives

$$D_{KL}(p\|q)=\sum_xp(x)\log p(x)-\sum_xp(x)\log q(x)=-H(p)+H(p,q),$$

so

$$H(p,q)=H(p)+D_{KL}(p\|q).$$

The cost of encoding $p$ with $q$ equals the irreducible uncertainty of the data plus the model's mismatch. Because $H(p)$ is constant with respect to the model, minimizing Cross-Entropy also minimizes KL, while requiring only $\log q$.


In [ ]:
# Verify H(p,q) = H(p) + D_KL(p || q) with dice
def cross_entropy(p, q, eps=1e-12):
    """Cost of encoding p using q."""
    return -(p * torch.log(q + eps)).sum().item()

def kl_divergence(p, q, eps=1e-12):
    """D_KL(p || q)"""
    return (p * (torch.log(p + eps) - torch.log(q + eps))).sum().item()

p = torch.tensor([0.4, 0.3, 0.2, 0.1])  # true distribution
q_good = torch.tensor([0.35, 0.35, 0.2, 0.1])  # prediction close to p
q_bad = torch.tensor([0.1, 0.2, 0.3, 0.4])     # prediction far from p

print("True distribution p:", p.tolist())
print()

for name, q in [("q_good, close to p", q_good), ("q_bad, far from p", q_bad)]:
    H_p = entropy(p)
    H_pq = cross_entropy(p, q)
    kl = kl_divergence(p, q)
    lhs = H_pq
    rhs = H_p + kl
    print(f"{name}:")
    print(f"  q             = {[round(x, 2) for x in q.tolist()]}")
    print(f"  H(p)          = {H_p:.4f}")
    print(f"  H(p, q)       = {H_pq:.4f}")
    print(f"  D_KL(p || q)  = {kl:.4f}")
    print(f"  H(p)+D_KL     = {rhs:.4f}  (should equal H(p,q) = {lhs:.4f})")
    print()

print("Key observation: H(p,q) and H(p) + D_KL(p||q) are numerically identical.")
print("As q approaches p, D_KL falls; when q=p, D_KL=0 and H(p,q)=H(p).")


## 4. Cross-Entropy Loss for Language Models

Language-model training connects maximum likelihood, negative log-likelihood (NLL), Cross-Entropy, and KL. For a sequence $x_1,\ldots,x_T$, maximum likelihood maximizes $\prod_t p_\theta(x_t\mid x_{<t})$. Taking logs and negating gives

$$\mathcal{L}_{NLL}=-\sum_t\log p_\theta(x_t\mid x_{<t}).$$

At each position, teacher forcing treats the observed next token as a one-hot true distribution $p$ and the model prediction as $q$. Therefore

$$H(p,q)=-\log q(x_t)=-\log p_\theta(x_t\mid x_{<t}).$$

Since a one-hot distribution has $H(p)=0$,

$$\mathcal{L}_{NLL}=H(p,q)=D_{KL}(p\|q).$$

This is why `F.cross_entropy` directly implements the language-model training objective.


In [ ]:
# Verify with tiny numbers: NLL = H(p,q) = D_KL(p || q) when p is one-hot
logits = torch.tensor([[2.0, 1.0, 0.5, -0.5]])  # one position, four classes
target_id = 1  # correct Token

# Method 1: PyTorch cross_entropy, which contains NLL
ce_loss = F.cross_entropy(logits, torch.tensor([target_id])).item()

# Method 2: hand-calculate NLL = -log p(correct)
log_probs = F.log_softmax(logits, dim=-1)
nll = -log_probs[0, target_id].item()

# Method 3: construct one-hot p and calculate KL
p_onehot = torch.zeros(4)
p_onehot[target_id] = 1.0
q = F.softmax(logits, dim=-1)[0]
# KL(p || q) = sum p log(p/q); p is nonzero only at target_id, giving log(1/q[target]) = -log q[target]
kl = (p_onehot * (torch.log(p_onehot + 1e-12) - torch.log(q + 1e-12))).sum().item()

print(f"PyTorch cross_entropy: {ce_loss:.6f}")
print(f"Hand-calculated NLL:     {nll:.6f}")
print(f"KL(p || q):           {kl:.6f}")
print()
print("All three values are identical, verifying NLL = CE = KL in LM training.")
print("Key observation: H(p)=0 for one-hot p, so CE and KL have no additive constant here.")


## 5. Perplexity

Perplexity is

$$\text{PPL}=\exp(\text{CE}),$$

where CE is average test-set Cross-Entropy using natural logarithms. It can be read as the effective number of equally likely choices per position: PPL 1 means certainty, PPL $V$ matches a uniform vocabulary distribution, and PPL 10 resembles uncertainty among ten choices.

Lower is better only under the same dataset and tokenizer. Tokenization changes token granularity and therefore changes Perplexity, so values across different tokenizers are not directly comparable.


In [ ]:
# Demonstrate PPL on a simple sequence
torch.manual_seed(0)

# Assume vocabulary V=10 and model logits at four positions
V = 10
logits = torch.tensor([
    [3.0, 1.0, 0.5, -0.5, 0.0, 0.2, -1.0, 0.1, -0.3, 0.4],  # position 0
    [0.1, 4.0, 0.2, -1.0, 0.0, 0.1, 0.0, -0.5, 0.3, -0.2],  # position 1, confident
    [0.3, 0.2, 0.4, 0.1, 0.5, 0.2, 0.3, 0.4, 0.1, 0.5],     # position 2, uncertain
    [1.5, 0.5, 2.0, -0.5, 0.0, 0.3, -1.0, 0.2, -0.3, 0.1],  # position 3
])
targets = torch.tensor([0, 1, 4, 2])

# Compute CE
ce = F.cross_entropy(logits, targets).item()
ppl = math.exp(ce)

print(f"Vocabulary size V = {V}")
print(f"Sequence length T = {len(targets)}")
print(f"Mean CE = {ce:.4f}")
print(f"PPL       = exp(CE) = {ppl:.4f}")
print()
print(f"\nKey observations:")
print(f"  PPL=1 means perfect prediction; PPL={V} means blind guessing")
print(f"  Current PPL={ppl:.2f}, between the two")
print(f"  Position 2 is nearly uniform and contributes most CE; position 1 is confident and contributes least")
print()

# CE at each position
per_pos_ce = F.cross_entropy(logits, targets, reduction='none')
print("Per-position CE:", [round(x, 3) for x in per_pos_ce.tolist()])
print("Per-position PPL:", [round(math.exp(x), 2) for x in per_pos_ce.tolist()])
print()
print("Key observation: perplexity is exp(CE), an exponentiated form of Cross-Entropy.")
print("PPL is not comparable across datasets because different tokenizers change the CE scale.")


## 6. Gradient of Softmax and Cross-Entropy

Together, Softmax and Cross-Entropy have a simple gradient:

$$\frac{\partial L}{\partial z}=p-\text{one\_hot}(t).$$

For $L=-\log p_t$, the Softmax Jacobian is $\partial p_j/\partial z_i=p_j(\delta_{ij}-p_i)$. Applying the chain rule leaves only $j=t$ and simplifies to $p_i-\delta_{it}$.

Every component lies in $[-1,1]$ because probabilities lie in $[0,1]$. This bounded form supports stable classification training. `F.cross_entropy` fuses log-softmax and target selection without explicitly materializing probabilities, preserving both stability and the same gradient.


In [ ]:
# Verify the softmax plus CE gradient formula: dL/dz = p - one_hot(t)
logits = torch.tensor([2.0, 1.0, 0.5, -0.5], requires_grad=True)
target_id = 1

# Method 1: PyTorch autograd
loss = F.cross_entropy(logits.unsqueeze(0), torch.tensor([target_id]))
loss.backward()
auto_grad = logits.grad.clone()

# Method 2: hand-calculate p - one_hot(t)
with torch.no_grad():
    p = F.softmax(logits, dim=-1)
    one_hot = torch.zeros(4)
    one_hot[target_id] = 1.0
    manual_grad = p - one_hot

print(f"target_id = {target_id}")
print(f"Softmax output p:    {[round(x, 4) for x in p.tolist()]}")
print(f"one_hot(t):     {one_hot.tolist()}")
print()
print(f"PyTorch gradient:    {[round(x, 4) for x in auto_grad.tolist()]}")
print(f"Manual p-one_hot:    {[round(x, 4) for x in manual_grad.tolist()]}")
print()
print("The values match exactly, verifying dL/dz = p - one_hot(t).")
print()
print("Interpret the sign of each gradient component:")
for i in range(4):
    g = manual_grad[i].item()
    if i == target_id:
        print(f"  Position {i}, correct: gradient {g:+.4f} is negative, so gradient descent raises z and p")
    else:
        print(f"  Position {i}, wrong:   gradient {g:+.4f} is positive, so gradient descent lowers z and p")
print()
print("Key observation: every gradient component lies in [-1, 1], helping softmax plus CE remain stable.")


## 7. Probability and Information Theory Across LLMs

| Area | Core concept | Mathematical form | Intuition |
|:---|:---|:---|:---|
| Pretraining / SFT | Cross-Entropy | $-\log p_\theta(x_t\mid x_{<t})$ | increase correct-token probability |
| Evaluation | Perplexity | $\exp(\text{CE})$ | effective choices per position |
| Temperature | scaled Softmax | $\text{softmax}(z/T)$ | deterministic to uniform |
| top-k / top-p | truncate and renormalize | $p^{(S)}/\sum_{j\in S}p_j$ | restrict candidates |
| RLHF | KL penalty | $\beta D_{KL}(\pi_\theta\|\pi_{ref})$ | limit policy drift |
| DPO | log-ratio | $\beta\log(\pi_\theta/\pi_{ref})$ | classify preferences |
| Distillation | KL | $D_{KL}(p_{teacher}\|p_{student})$ | imitate teacher distribution |
| MoE routing | categorical distribution | $\text{softmax}(W_{gate}x)$ | assign experts |
| MoE balancing | Entropy / KL auxiliary loss | $\alpha\,\text{aux}(\bar p)$ | balance expert use |

KL direction changes behavior. Distillation usually uses teacher-to-student KL so the student covers the teacher's possible outputs. DPO's policy/reference log-ratio acts as an implicit KL constraint. MoE balancing losses may use Entropy, KL to uniform, or utilization variance; each constrains the shape of a categorical distribution. When reading a new paper, ask: which distribution is it changing, and which divergence is it minimizing?


## 8. MoE Routing and Load Balancing

An MoE router maps each token to `num_experts` scores, selects the top-$k$, and mixes their outputs with Softmax weights. It therefore emits a categorical distribution over experts, mathematically identical to a language-model head over tokens.

Unconstrained routing may overload a few experts. Common auxiliary losses include:

1. **Entropy**: $L_{aux}=-H(\bar p)$ encourages a uniform batch-average route distribution.
2. **KL**: $L_{aux}=D_{KL}(\bar p\|u)$ explicitly matches a uniform target $u$.
3. **Switch Transformer**: $L_{aux}=N\sum_i f_ip_i$ encourages both selection frequency $f_i$ and mean routing probability $p_i$ toward $1/N$.

All three constrain the categorical distribution toward balanced expert use. A Softmax router models competition among mutually selected experts; independent sigmoid gates express a different routing assumption.


In [ ]:
# Demonstrate router categorical output with a tiny MoE
import torch
import torch.nn.functional as F

torch.manual_seed(42)

d_model = 8
num_experts = 4
top_k = 2

# Minimal router: one linear layer
W_gate = torch.randn(d_model, num_experts)

# Simulate three Tokens
tokens = torch.randn(3, d_model)
gate_logits = tokens @ W_gate  # [3, num_experts]
gate_probs = F.softmax(gate_logits, dim=-1)

print("=== Categorical distribution output by the router ===")
for i in range(3):
    print(f"token {i}: gate_logits = {[round(x, 3) for x in gate_logits[i].tolist()]}")
    print(f"         gate_probs   = {[round(x, 3) for x in gate_probs[i].tolist()]}")
    topk_vals, topk_idx = torch.topk(gate_logits[i], top_k)
    topk_weights = F.softmax(topk_vals, dim=-1)
    print(f"         top-{top_k} selection: experts {topk_idx.tolist()}, weights {[round(x, 3) for x in topk_weights.tolist()]}")
    print()

# Calculate mean routing distribution across the batch to demonstrate load-balancing loss
mean_routing = gate_probs.mean(dim=0)  # [num_experts]
uniform = torch.ones(num_experts) / num_experts

# Form 1: negative entropy
neg_entropy = -(mean_routing * torch.log(mean_routing + 1e-12)).sum().item()

# Form 2: KL(mean_p || uniform)
kl_to_uniform = (mean_routing * (torch.log(mean_routing + 1e-12) - torch.log(uniform + 1e-12))).sum().item()

print("=== Two forms of load-balancing loss ===")
print(f"Mean batch routing distribution: {[round(x, 4) for x in mean_routing.tolist()]}")
print(f"Uniform distribution:           {[round(x, 4) for x in uniform.tolist()]}")
print(f"-H(mean distribution)           = {neg_entropy:.4f}  (smaller is closer to uniform)")
print(f"KL(mean || uniform)             = {kl_to_uniform:.4f}  (smaller is closer to uniform)")
print()
print("Key observation: both loss forms measure how far the mean routing distribution departs from uniform.")
print("Minimizing either encourages uniform routing and prevents a few experts from overloading.")


## Summary

- [ ] Softmax converts logits into a categorical distribution.
- [ ] Subtracting the maximum gives numerically stable Softmax and logsumexp.
- [ ] Temperature divides logits: $T\to0$ approaches greedy and $T\to\infty$ approaches uniform.
- [ ] top-k truncates by count; top-p truncates by cumulative probability.
- [ ] Entropy measures uncertainty, Cross-Entropy encoding cost, and KL distribution mismatch.
- [ ] $H(p,q)=H(p)+D_{KL}(p\|q)$ connects all three.
- [ ] Under teacher forcing, one-hot targets make NLL = CE = KL.
- [ ] PPL = exp(CE) and is comparable only with the same data and tokenizer.
- [ ] The Softmax + CE gradient is $p-\text{one\_hot}(t)$.
- [ ] Training, alignment, distillation, decoding, evaluation, and MoE reuse the same objects.
- [ ] MoE routing is categorical, and balancing losses constrain its distribution.

In one sentence: categorical distributions and the Entropy–Cross-Entropy–KL trio provide one shared mathematical map for language-model training, inference, alignment, evaluation, and routing.


## Exercises

> You may use AI to help explain ideas, but it is not recommended to have AI "complete the exercise for you".

**Exercise 1: Compute entropy by hand**

Given the distribution $p = [0.5, 0.25, 0.125, 0.125]$, compute $H(p)$ by hand.

Hint: $H(p) = -\sum p_i \log p_i$ using the natural logarithm.


In [ ]:
import math
import torch

p = torch.tensor([0.5, 0.25, 0.125, 0.125])

# TODO: hand-calculate H(p)
manual_H = None  # enter the hand-calculated result

# Verify with PyTorch
torch_H = -(p * torch.log(p)).sum().item()

assert manual_H is not None, 'Please replace the placeholder before running the assertion.'
assert abs(manual_H - torch_H) < 1e-6, f"Answer should be {torch_H:.6f}; you got {manual_H}"

print(f"✅ Exercise 1 passed")
print(f"   H(p) = {manual_H:.4f}")
print(f"   theoretical maximum log(4) = {math.log(4):.4f}")
print(f"   p is more certain than uniform, so its entropy is below log(4)")


**Exercise 2: Implement a numerically stable log-softmax**

Given a set of logits (including large numbers), implement a numerically stable log-softmax using the logsumexp trick. You may not call `F.log_softmax` directly.

Hint: $\log p_i = z_i - \text{logsumexp}(z)$, where $\text{logsumexp}(z) = c + \log \sum_j e^{z_j - c}$ and $c = \max_j z_j$.


In [ ]:
import torch
import torch.nn.functional as F

big_logits = torch.tensor([1000.0, 1001.0, 1002.0, 999.0])

def stable_log_softmax(z):
    """Numerically stable log-softmax without calling F.log_softmax."""
    # TODO: implement with the logsumexp trick
    c = z.max()
    logsumexp = None  # enter c + log(sum(exp(z - c)))
    log_probs = None  # enter z - logsumexp
    return log_probs

manual_lp = stable_log_softmax(big_logits)
torch_lp = F.log_softmax(big_logits, dim=-1)

assert manual_lp is not None, 'Please replace the placeholder before running the assertion.'
assert torch.allclose(manual_lp, torch_lp, atol=1e-5), \
    f"Result differs from PyTorch\nYou got: {manual_lp.tolist()}\nPyTorch: {torch_lp.tolist()}"

print("Exercise 2 passed:")
print(f"   log_softmax = {[round(x, 4) for x in manual_lp.tolist()]}")
print(f"   corresponding probabilities = {[round(x, 4) for x in torch.exp(manual_lp).tolist()]}")
print(f"   probability sum = {torch.exp(manual_lp).sum().item():.6f}")
print("   Key: subtract max before exp to prevent large values such as 1,000 from overflowing.")


**Exercise 3: Measure PPL in practice**

Given a set of logits and the corresponding targets, compute the average CE and PPL, and explain the meaning of PPL.

Hint: PPL = exp(average CE); use `F.cross_entropy` to compute CE and `math.exp` to compute PPL.


In [ ]:
import math
import torch
import torch.nn.functional as F

V = 8
logits = torch.tensor([
    [2.0, 1.0, 0.5, -0.5, 0.0, 0.2, -1.0, 0.1],
    [0.1, 0.2, 0.3, 0.1, 0.2, 0.3, 0.1, 0.2],  # highly uncertain
    [5.0, 0.1, 0.0, -1.0, 0.2, -0.5, 0.1, -0.3],  # highly confident
])
targets = torch.tensor([0, 4, 0])

# TODO: calculate mean CE and PPL
ce = None       # use F.cross_entropy
ppl = None      # use math.exp(ce)

assert ce is not None and ppl is not None, 'Please replace the placeholder before running the assertion.'

# Verify
expected_ce = F.cross_entropy(logits, targets).item()
expected_ppl = math.exp(expected_ce)
assert abs(ce - expected_ce) < 1e-6
assert abs(ppl - expected_ppl) < 1e-3

print(f"✅ Exercise 3 passed")
print(f"   mean CE = {ce:.4f}")
print(f"   PPL    = {ppl:.4f}")
print(f"   vocabulary V = {V}")
print()
print(f"Exercise 1 passed:")
print(f"   PPL={ppl:.2f} means the model is as uncertain as choosing uniformly among about {ppl:.1f} candidates")
print(f"   Position 2 with logit 5.0 is confident and contributes least CE; position 1 is nearly uniform and contributes most")
print(f"   PPL near V={V} means near-random guessing; PPL near 1 means nearly all predictions are correct")
